In [1]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
%run $header_path

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
initalizing spark nlp session w gpu


In [2]:
accented = r"àâäáçèéêëîïìíôöòóùûüúÿ"
plain     = r"aaaaceeeeiiiioooouuuuy"
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\raphael.wargon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Fichier DGPIE

## Merge organisations

In [3]:
all_anr = spark.read.csv('file:///' + save_path + "ods_france2030-partenaires.csv", sep = ";", header = True)

In [4]:
df_inst = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'institutions.parquet')
           .withColumn('inst_id', func.regexp_replace(func.col('id'), 'https://openalex.org/', ''))
          )

In [1420]:
search="Écosystème"

In [1421]:
search = search.lower()
(df_inst.filter( (func.lower(func.col('display_name')).contains(search))
               | ( func.array_max(func.transform(func.col('display_name_alternatives'), lambda x: (func.lower(x).contains(search)).cast('int'))) ==1)
              | ( func.array_max(func.transform(func.col('display_name_acronyms'), lambda x: (func.lower(x).contains(search)).cast('int'))) ==1)
              | ( func.array_max(func.transform(func.col('associated_institutions.display_name'), lambda x: (func.lower(x).contains(search)).cast('int'))) ==1)

               )
.select('display_name', func.regexp_replace(func.col('id'), 'https://openalex.org/', '').alias('inst_id'), "type", 'country_code', func.col('associated_institutions.display_name')) ).show(100, truncate = False)

+------------------------------------------------------------+-----------+---------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [1333]:
all_anr.show()

+------------+---------------+------------------------+------------------+------------------+--------------------+------------------------------+------------------+-------------+--------------------+--------------+---------------------------+---------------------+--------------------------+------------------+
| Eotp Projet|Eotp Partenaire|Etablissement Partenaire| Partenaire Région|Partenaire Secteur|Partenaire Catégorie|Tag Etablissement Coordinateur|Total Conventionné|     Acronyme|    Region du projet|Code_Region_TG|Année de contractualisation|Nom d'action détaillé|Lien vers le site de l'ANR|code_region_projet|
+------------+---------------+------------------------+------------------+------------------+--------------------+------------------------------+------------------+-------------+--------------------+--------------+---------------------------+---------------------+--------------------------+------------------+
|10-AIRT-0001|10-AIRT-0001-04|                  Aurock|         Occ

In [8]:
df_inst.show()

+-----------------------+--------------+------------+--------------------+--------------------+--------------------+---------------------+-------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+-----------+
|associated_institutions|cited_by_count|country_code|      counts_by_year|        created_date|        display_name|display_name_acronyms|display_name_alternatives|                 geo|        homepage_url|                  id|                 ids| image_thumbnail_url|           image_url|is_super_system|             lineage|        repositories|               roles|                 ror|       summary_s

In [17]:
merge_inst = (all_anr
              .withColumn('tokens', func.split( func.translate(func.regexp_replace(
                  func.lower(func.col('Etablissement Partenaire')), r"[\-'’]", ' '), accented, plain), ' '))
              .withColumn('tokens', func.array_compact(func.transform(func.col('tokens'), lambda x: func.when(func.length(x)>1, 
                                                                                                              func.regexp_replace(x, r'[\W]', '') ))))
              .withColumn('tokens', func.array_except(func.col('tokens'), func.array([func.lit(word) for word in stopwords.words('french')]) ) )
              .withColumn('normalized_name', func.concat_ws(' ', func.col('tokens')))
                    .withColumn('token', func.explode(func.col('tokens')))
              .filter(func.length('token')>=3)
              .join(df_inst
                    .select('inst_id', func.explode( func.array_append(func.col('display_name_alternatives'), func.col('display_name'))).alias('display_name'))
                    .withColumn('tokens_oa', func.split( func.translate(func.regexp_replace(
                        func.lower(func.col('display_name')), r"[\-'’]", ' '), accented, plain), ' '))
                    .withColumn('tokens_oa', func.array_compact(func.transform(func.col('tokens_oa'), lambda x: func.when(func.length(x)>1,  func.regexp_replace(x, r'[\W]', '')))))
                    .withColumn('tokens_oa', func.array_except(func.col('tokens_oa'), func.array([func.lit(word) for word in stopwords.words('french')]) ) )
                    .withColumn('normalized_name_oa', func.concat_ws(' ', func.col('tokens_oa')))
                    .withColumn('token', func.explode(func.col('tokens_oa')))
                    , on = 'token', how = 'left')
              .withColumn('name_match', (func.col('normalized_name') == func.col('normalized_name_oa')).cast('int'))
             )
merge_inst.cache()
merge_inst.show()



+------------+------------+---------------+------------------------+-----------------+------------------+--------------------+------------------------------+------------------+-------------+----------------+--------------+---------------------------+---------------------+--------------------------+------------------+--------------------+-----------------+-----------+--------------------+--------------------+--------------------+----------+
|       token| Eotp Projet|Eotp Partenaire|Etablissement Partenaire|Partenaire Région|Partenaire Secteur|Partenaire Catégorie|Tag Etablissement Coordinateur|Total Conventionné|     Acronyme|Region du projet|Code_Region_TG|Année de contractualisation|Nom d'action détaillé|Lien vers le site de l'ANR|code_region_projet|              tokens|  normalized_name|    inst_id|        display_name|           tokens_oa|  normalized_name_oa|name_match|
+------------+------------+---------------+------------------------+-----------------+------------------+-------

In [18]:
merge_inst.groupBy('name_match').count().show()

+----------+--------+
|name_match|   count|
+----------+--------+
|      NULL|    3329|
|         1|   18656|
|         0|17921712|
+----------+--------+



In [19]:
merge_inst.groupBy('normalized_name').agg(func.max('name_match').alias('name_match')).groupBy('name_match').count().show()

+----------+-----+
|name_match|count|
+----------+-----+
|      NULL|  309|
|         1|  383|
|         0| 1276|
+----------+-----+



In [38]:
merge_inst_round2 = (merge_inst
                     .withColumn('small_change_match',
                                 func.when( func.regexp_replace( func.col('normalized_name'), 'university', 'universite' ) ==  
                                            func.regexp_replace( func.col('normalized_name_oa'), 'university', 'universite' ), 1)
                                 .when( func.regexp_replace( func.col('normalized_name'), 'institute', 'institut' ) ==  
                                        func.regexp_replace( func.col('normalized_name_oa'), 'institute', 'institut' ), 1)
                                 .when( func.regexp_replace( func.col('normalized_name'), 'chru ', 'chu ' ) ==  
                                        func.regexp_replace( func.col('normalized_name_oa'), 'chru ', 'chu ' ), 1)
                                 .when( func.regexp_replace( func.col('Etablissement Partenaire'), '-', '' ) ==  
                                        func.regexp_replace( func.col('display_name'), '-', '' ), 1)

                                 .when( func.concat_ws(' ', func.transform(func.col('tokens'), lambda x: func.regexp_replace(x, r'[sx]\z', ''))) == 
                                        func.concat_ws(' ', func.transform(func.col('tokens_oa'), lambda x: func.regexp_replace(x, r'[sx]\z', ''))), 1)
                                 .otherwise(0))
                    )

In [61]:
unmatched = (merge_inst_round2
             .fillna(0, ['name_match','small_change_match']).withColumn('matched', func.greatest(func.col('name_match'), func.col('small_change_match')))
             .withColumn('inst_id', func.when(func.col('matched')==1, func.col('inst_id')))
             .withColumn('display_name', func.when(func.col('matched')==1, func.col('display_name')))
             .select('Etablissement Partenaire', "matched", 'display_name', 'inst_id').distinct()
       )
unmatched.cache()
unmatched.show()

+------------------------+-------+--------------------+-----------+
|Etablissement Partenaire|matched|        display_name|    inst_id|
+------------------------+-------+--------------------+-----------+
|    National Institut...|      1|National Institut...|I4210118971|
|                  Airbus|      0|                NULL|       NULL|
|    Continental Autom...|      0|                NULL|       NULL|
|    Peugeot Citroen A...|      0|                NULL|       NULL|
|    Centre Hospitalie...|      0|                NULL|       NULL|
|          GRAINE DE CHOC|      0|                NULL|       NULL|
|           Orsay Physics|      0|                NULL|       NULL|
|           ALGUE SERVICE|      0|                NULL|       NULL|
|     Universität Hamburg|      0|                NULL|       NULL|
|          GIE Pharmimage|      0|                NULL|       NULL|
|    JESSICA France – ...|      0|                NULL|       NULL|
|     Université de Lille|      1| Université de

In [89]:
unmatched = spark.read.csv('file:///C:/Users/raphael.wargon/Downloads/anr_etablissement_partenaires_a_matcher - anr_etablissement_partenaires_a_matcher.csv', header = True).drop('c_0')

In [90]:
unmatched = (unmatched
             .groupBy('Etablissement Partenaire')
             .agg(func.max(func.col('matched')).alias('matched'),
                  func.concat_ws(',', func.collect_set('display_name')).alias('display_name'),
                  func.concat_ws(',', func.collect_set('inst_id')).alias('inst_id'))
            )

In [91]:
unmatched.show()

+------------------------+-------+--------------------+-----------+
|Etablissement Partenaire|matched|        display_name|    inst_id|
+------------------------+-------+--------------------+-----------+
|                    2GRE|      0|                    |           |
|                 3D PLUS|      0|                    |           |
|    3mensio Medical I...|      0|                    |           |
|                     A3I|      0|                    |           |
|              AB SCIENCE|      1| AB Science (France)|I4210091652|
|             AB SCIEX LP|      1|      Sciex (Canada)|I4210131534|
|               ABBELIGHT|      0|                    |           |
|            ABYS MEDICAL|      0|                    |           |
|                     ACB|      0|                    |           |
|                   ACOME|      0|                    |           |
|    ACTA - Les Instit...|      1|Acta les institut...|I4210104339|
|                 ACTALIA|      1|    Actalia (F

In [92]:
unmatched = (unmatched
             .withColumn('normalized_name', func.lower(func.col('Etablissement Partenaire')))
             .join(df_inst.select(func.col('display_name').alias('display_name_'), func.col('inst_id').alias('inst_id_'))
                   .withColumn('normalized_name', func.lower(func.regexp_replace(func.col('display_name_'), ' (France)', '')))
                   , on = 'normalized_name', how = "left")
            )
unmatched.filter(func.col('inst_id_').isNotNull()).count()

174

In [93]:
unmatched = (unmatched
             .withColumn('display_name', func.when(func.col('display_name') != '', func.col('display_name')))
             .withColumn('inst_id', func.when(func.col('inst_id') != '', func.col('inst_id')))

             .groupBy('Etablissement Partenaire')
             .agg(func.max(func.when( (func.col('inst_id') != '') | (func.col('inst_id_') != ''), 1).otherwise(0)).alias('matched'),
                  func.concat_ws(',', func.collect_set(func.coalesce(func.col('display_name'), func.col('display_name_')))).alias('display_name'),
                  func.concat_ws(',', func.collect_set(func.coalesce(func.col('inst_id'), func.col('inst_id_')))).alias('inst_id'))
            )


In [94]:
unmatched.groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1|  512|
|      0| 1462|
+-------+-----+



In [96]:
df_inst.filter(func.col('Etablissement Partenaire')=="Bureau Veritas").show()

+------------------------+-------+------------+-------+
|Etablissement Partenaire|matched|display_name|inst_id|
+------------------------+-------+------------+-------+
|          Bureau Veritas|      0|            |       |
+------------------------+-------+------------+-------+



In [95]:
unmatched.toPandas().to_csv(save_path + '/anr_etablissement_partenaires_a_matcher.csv')

## matched finished by hand

In [1427]:
matched_etablissement_partenaires = spark.read.csv("file:///" + save_path + "anr_match_etablissement_partenaires.csv", header = True).drop('_c0')

In [1428]:
matched_etablissement_partenaires.show()

+------------------------+-------+--------------------+-----------+
|Etablissement Partenaire|matched|        display_name|    inst_id|
+------------------------+-------+--------------------+-----------+
|                    2GRE|      0|                NULL|       NULL|
|                 3D PLUS|      0|                NULL|       NULL|
|    3mensio Medical I...|      0|                NULL|       NULL|
|                     A3I|      0|                NULL|       NULL|
|              AB SCIENCE|      1| AB Science (France)|I4210091652|
|             AB SCIEX LP|      1|      Sciex (Canada)|I4210131534|
|               ABBELIGHT|      0|                NULL|       NULL|
|            ABYS MEDICAL|      0|                NULL|       NULL|
|                     ACB|      0|                NULL|       NULL|
|                   ACOME|      0|                NULL|       NULL|
|    ACTA - Les Instit...|      1|Acta les institut...|I4210104339|
|                 ACTALIA|      1|    Actalia (F

In [1429]:
all_anr = (all_anr
           .join(matched_etablissement_partenaires, on = 'Etablissement Partenaire', how = 'left')
          )

In [1437]:
all_anr.show()

+------------------------+------------+---------------+------------------+------------------+--------------------+------------------------------+------------------+-------------+--------------------+--------------+---------------------------+---------------------+--------------------------+------------------+-------+--------------------+-----------+
|Etablissement Partenaire| Eotp Projet|Eotp Partenaire| Partenaire Région|Partenaire Secteur|Partenaire Catégorie|Tag Etablissement Coordinateur|Total Conventionné|     Acronyme|    Region du projet|Code_Region_TG|Année de contractualisation|Nom d'action détaillé|Lien vers le site de l'ANR|code_region_projet|matched|        display_name|    inst_id|
+------------------------+------------+---------------+------------------+------------------+--------------------+------------------------------+------------------+-------------+--------------------+--------------+---------------------------+---------------------+--------------------------+-----

In [1431]:
all_anr.fillna(0, ['matched']).groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      0| 1740|
|   NULL|  102|
|      1|11892|
+-------+-----+



In [1445]:
all_anr.filter(func.lower(func.col('Acronyme')).contains('labex')).show()

+------------------------+------------+---------------+--------------------+------------------+--------------------+------------------------------+------------------+----------+--------------------+--------------+---------------------------+---------------------+--------------------------+------------------+-------+--------------------+-----------+
|Etablissement Partenaire| Eotp Projet|Eotp Partenaire|   Partenaire Région|Partenaire Secteur|Partenaire Catégorie|Tag Etablissement Coordinateur|Total Conventionné|  Acronyme|    Region du projet|Code_Region_TG|Année de contractualisation|Nom d'action détaillé|Lien vers le site de l'ANR|code_region_projet|matched|        display_name|    inst_id|
+------------------------+------------+---------------+--------------------+------------------+--------------------+------------------------------+------------------+----------+--------------------+--------------+---------------------------+---------------------+--------------------------+--------

In [1450]:
all_anr_wnames = spark.read.csv('file:///' + save_path + "ods_france2030-projets.csv", sep = ";", header = True)

In [1451]:
all_anr_wnames.show()

+------------+------------+--------------------+--------------------+-------------+---+-----------+--------------------+--------------+------------------+-------+--------------------+-------------------+----------+-----------------+----------+--------------+---------------------------+-------------+---------------------+--------------------------+
| Eotp Projet|    Acronyme|              Résumé|  Projet Nom Complet|   Thématique|PIA|Code Action|          Action Nom|Date Contract.|Total Conventionné|   Etat|    Region du projet|                RST| Date Init|Date Début Projet|  Date Fin|Code_Region_TG|Année de contractualisation|Type d'Action|Nom d'action détaillé|Lien vers le site de l'ANR|
+------------+------------+--------------------+--------------------+-------------+---+-----------+--------------------+--------------+------------------+-------+--------------------+-------------------+----------+-----------------+----------+--------------+---------------------------+-------------+

In [1452]:
anr_to_merge_au = (all_anr
                   .join(all_anr_wnames
                         .select('Eotp Projet', 'Etat', 'RST', 'Date Début Projet', 'Date Fin', "Type d'Action"),
                         on = "Eotp Projet", how = 'left')
                  .withColumn('normalized_name', func.translate(func.lower(func.col('RST')), accented, plain))
                  )

In [1453]:
anr_to_merge_au.show()

+------------+------------------------+---------------+------------------+------------------+--------------------+------------------------------+------------------+-------------+--------------------+--------------+---------------------------+---------------------+--------------------------+------------------+-------+--------------------+-----------+-------+--------------------+-----------------+----------+-------------+--------------------+
| Eotp Projet|Etablissement Partenaire|Eotp Partenaire| Partenaire Région|Partenaire Secteur|Partenaire Catégorie|Tag Etablissement Coordinateur|Total Conventionné|     Acronyme|    Region du projet|Code_Region_TG|Année de contractualisation|Nom d'action détaillé|Lien vers le site de l'ANR|code_region_projet|matched|        display_name|    inst_id|   Etat|                 RST|Date Début Projet|  Date Fin|Type d'Action|     normalized_name|
+------------+------------------------+---------------+------------------+------------------+-----------------

In [1535]:
all_anr_wnames.filter(func.col('eotp projet').contains('LAB')).show(truncate = False)

+------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [1538]:
all_anr_wnames.filter(func.col('eotp projet').contains('LAB')).select('acronyme').distinct().count()

171

In [3]:
au_inst_period = spark.read.format('parquet').load('file:\\' + int_save_path + 'panel_non_sphericised.parquet')

In [1527]:
au_inst_period.filter(func.lower(func.col('name')).rlike('labex|labx')).count()

943

In [1529]:
au_inst_period.filter(func.lower(func.col('name')).rlike('labex|labx')).summary().show()

+-------+-----------+-----------+------------------+--------------------+--------------------+-------------------+--------------------+------------------+-----------------+-----------------+-------------------+-----------------------+-----------------------+-----------------------+-----------------------+----------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+---------------------+--------------------+------------------+--------------------+------------------+----------------------+-------------------+-------------------------+---------------------+---------------------------+------------------+------------------+------------------+--------+--------------------+---------------+-------------------+-------------------+-----------------+----+------------------+-------------------+--------------+------------------+-------------------+---------+--------------------+-----+-----------------

In [1485]:
inst = spark.read.parquet('file:\\' + int_save_path + 'inst_fr.parquet')

In [1528]:
inst.filter(func.col('cnrs')==1 ).filter(func.col('type')=="facility").show()

+-----------+--------------------+--------+--------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+----+----+-------------------+-------------------+---------+---------+---------------+-----------+--------------+
|    inst_id|                name|    type|          city|        homepage_url|         topic_share|          main_topic|              parent|uni_pub|           parent_id|cnrs|idex|total_idex_dotation|            type_fr|  secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+-----------+--------------------+--------+--------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+----+----+-------------------+-------------------+---------+---------+---------------+-----------+--------------+
|I4210101037|Laboratoire de Ph...|facility|       Dunkirk|https://lpca.univ...|[{Chemistry, 0.00...|           Chemistry|[{I2802487185, Un...|   

In [1539]:
inst.filter(func.col('name').contains("Droits, Pouvoirs ")).show()

+-------+----+----+----+------------+-----------+----------+------+-------+---------+----+----+-------------------+-------+-------+---------+---------------+-----------+--------------+
|inst_id|name|type|city|homepage_url|topic_share|main_topic|parent|uni_pub|parent_id|cnrs|idex|total_idex_dotation|type_fr|secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+-------+----+----+----+------------+-----------+----------+------+-------+---------+----+----+-------------------+-------+-------+---------+---------------+-----------+--------------+
+-------+----+----+----+------------+-----------+----------+------+-------+---------+----+----+-------------------+-------+-------+---------+---------------+-----------+--------------+



In [1533]:
inst.filter(func.col('name').contains('ARBRE')).show(truncate = False)

+-----------+-----------+--------+----------+---------------------------------+---------------------------------------------------------------------------------------+-----------+----------------------------------------------------------------------------------------------------------+-------+------------------------+----+----+-------------------+-------------------+-------+---------+---------------+-----------+--------------+
|inst_id    |name       |type    |city      |homepage_url                     |topic_share                                                                            |main_topic |parent                                                                                                    |uni_pub|parent_id               |cnrs|idex|total_idex_dotation|type_fr            |secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+-----------+-----------+--------+----------+---------------------------------+-----------------------------------------------------------

In [1487]:
au_inst = (au_inst_period.select('author_id', 'inst_id', 'author_name')
          .withColumn('normalized_name_oa', func.translate(func.lower(func.col('author_name')), accented, plain))
           .join(inst.select('inst_id', 'parent.parent_id'), on = 'inst_id', how = 'left')
           .withColumn('inst_id', func.explode(func.array_append(func.col('parent_id'), func.col('inst_id'))))
           .distinct())

In [1501]:
au_inst.filter(func.col('author_id')=="A5022868858").show()

+-----------+-----------+----------------+------------------+--------------------+
|    inst_id|  author_id|     author_name|normalized_name_oa|           parent_id|
+-----------+-----------+----------------+------------------+--------------------+
| I102197404|A5022868858|Florence Geneste|  florence geneste|        [I102197404]|
|I4210128300|A5022868858|Florence Geneste|  florence geneste|       [I1294671590]|
|I1294671590|A5022868858|Florence Geneste|  florence geneste|[I1294671590, I28...|
|  I28221208|A5022868858|Florence Geneste|  florence geneste|[I1294671590, I28...|
|  I56067802|A5022868858|Florence Geneste|  florence geneste|[I1294671590, I28...|
|I1294671590|A5022868858|Florence Geneste|  florence geneste|[I1294671590, I27...|
|I1294671590|A5022868858|Florence Geneste|  florence geneste|       [I1294671590]|
|I4210128300|A5022868858|Florence Geneste|  florence geneste|[I1294671590, I28...|
|  I56067802|A5022868858|Florence Geneste|  florence geneste|         [I56067802]|
| I2

In [1467]:
anr_to_merge_au.select('normalized_name').distinct().count()

1826

In [1506]:
dash_expr = r'[\u00AD\u2010\u2011\u2012\u2013\u2014\u2015\u2212\uFE58\uFE63\uFF0D\-]'
merge_ods = (anr_to_merge_au
             #.withColumn('inst_id', func.collect_set('inst_id').over(Window.partitionBy('Eotp Projet')))
             #.withColumn('inst_id', func.explode(func.col('inst_id')))
             .join(au_inst, on ='inst_id')
             .withColumn('matched_au', func.when( func.col('normalized_name')==func.col('normalized_name_oa'), 1)
                         .when(func.regexp_replace(func.col('normalized_name'), dash_expr, ' ') ==  func.regexp_replace(func.col('normalized_name_oa'), dash_expr, ' '), 1)
                         .when(func.reverse(func.split(func.col('normalized_name'), " " )) == func.split(func.col('normalized_name_oa'), " " ), 1)
                         .when(func.regexp_replace(func.col('normalized_name'), r'\-[a-z]+\z', '') ==  func.regexp_replace(func.col('normalized_name_oa'),  r'\-[a-z]+\z', ''), 1)
                         .when(func.regexp_replace(func.col('normalized_name'), r'\s[A-Z]\.', '') ==  func.regexp_replace(func.col('normalized_name_oa'),   r'\s[A-Z]\.', ''), 1)
                         .otherwise(0)
                        )
            )
merge_ods.count()

272555568

In [1507]:
merge_ods.groupBy('normalized_name').agg(func.max('matched_au').alias('matched_au')).groupBy('matched_au').count().show()

+----------+-----+
|matched_au|count|
+----------+-----+
|         1| 1287|
|         0|  515|
+----------+-----+



## Merge the rest by name

In [1520]:
unmatched_authors = merge_ods.groupBy('RST').agg(func.max('matched_au').alias('matched_au')).filter('matched_au=0')

In [1521]:
unmatched_authors.toPandas().to_csv(save_path + '/anr_ods_noms_a_matcher.csv')

In [1522]:
matched_authors_by_code = (merge_ods.filter('matched_au=1').groupBy("RST").agg(func.collect_set('author_id')).alias('author_id'))

In [1523]:
matched_authors_by_code.show()

+--------------------+----------------------+
|                 RST|collect_set(author_id)|
+--------------------+----------------------+
|       ARNAUD RENARD|         [A5007726652]|
|       AUBERT PASCAL|  [A5103940991, A51...|
|Abdel-Illah MOUADDIB|         [A5019910212]|
|Abdeljalil ABBAS ...|         [A5087519995]|
|Abderrahmane KHEDDAR|         [A5067136188]|
|       Adrien BIDAUD|         [A5011161966]|
|        Adrien LEBRE|         [A5072415645]|
|      Adrien TOUTANT|         [A5089148274]|
|       Adélie POMADE|         [A5018562735]|
|        Agathe EUZEN|         [A5087249119]|
| Agnès DUPRET-BORIES|         [A5069683945]|
|Agnès FICHARD-CAR...|  [A5030958164, A50...|
|  Agnès HELME-GUIZON|         [A5081406986]|
|    Agnès LABOUDIGUE|         [A5104993297]|
|      Agnès LE MASLE|         [A5036958569]|
|Agnès MARSSET-BAG...|         [A5053802330]|
|        Agnès NADJAR|         [A5031913251]|
|       Ahidjo AYOUBA|         [A5087573541]|
|     Ahmed BOUNEKKAR|         [A5

In [1540]:
matched_authors_by_hand = spark.read.csv(save_path+ '/anr_ods_noms_matchés.csv', header = True).filter('matched_au=1')

In [1542]:
ods_fully_merged = (anr_to_merge_au
                    .join(matched_authors_by_code.unionAll(matched_authors_by_hand.select('RST', func.array(func.col('author_id')).alias('author_id'))),
                          on = 'RST', how = 'left')
                   )

In [1544]:
ods_fully_merged.write.parquet(save_path + '/ods_france2030-fully_matched.parquet')

# Fichiers DOS/DGDS

## merge authors directly

In [4]:
df_authors = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'authors.parquet')
              .withColumn('max_updated_date', func.max(func.col('updated_date')).over(Window.partitionBy('id')))
              .filter(func.col('updated_date')== func.col('max_updated_date'))
              .withColumn('split_name', func.split(func.col('display_name'), ' '))
             .withColumn('first_name', func.element_at(func.col('split_name'), 1))
              .withColumn('last_name', func.concat_ws(' ', func.array_except(func.col('split_name'), func.array(func.col('first_name')))))
              .select(func.regexp_replace(func.col('id'), 'https://openalex.org/', '').alias('author_id'), 'display_name',
                      'orcid', 'display_name_alternatives', 'affiliations')
             )


In [5]:
df_inst = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'institutions.parquet')
           .withColumn('inst_id', func.regexp_replace(func.col('id'), 'https://openalex.org/', ''))
          )

In [6]:
au_level_char= (df_authors
                .join(spark.read.parquet('file:\\' + int_save_path + 'au_level_char.parquet'), on = 'author_id', how = 'inner')
               )
#au_level_char.cache()
#au_level_char.show()

In [7]:
dash_expr = r'[\u00AD\u2010\u2011\u2012\u2013\u2014\u2015\u2212\uFE58\uFE63\uFF0D\-]'

au_level_char_w_inst = (au_level_char
                        .withColumn('affiliations', func.explode(func.col('affiliations.institution')))
                        .withColumn('inst_id', func.regexp_replace(func.col('affiliations.id'), 'https://openalex.org/', '') )
                        .join(df_inst.select('inst_id', 'geo.city', 'associated_institutions', func.col('display_name_acronyms').alias('acronyms') ), on = 'inst_id', how = 'left')
                        .groupBy(*[col for col in au_level_char.columns if col !='affiliations'])
                        .agg( func.collect_set("affiliations").alias('affiliations'),
                             func.collect_set('city').alias('city'),
                             func.first('associated_institutions').alias('associated_institutions'),
                            func.flatten(func.collect_set('acronyms')).alias('acronyms'))
                        .withColumn('normalized_name', func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', '')))
                        .withColumn("affiliations",
                             func.transform(func.col('affiliations.display_name'), lambda x :
                                            func.lower(func.regexp_replace(func.translate(x,  accented, plain), r'[^A-z\s\-]*', '')) ) )
                        .withColumn('associated_institutions',
                             func.transform(func.col('associated_institutions.display_name'), lambda x :
                                            func.lower(func.regexp_replace(func.translate(x,  accented, plain), r'[^A-z\s\-]*', '')) ))
                        .withColumn('acronyms',
                             func.transform(func.col('acronyms'), lambda x :
                                            func.lower(func.regexp_replace(func.translate(x,  accented, plain), r'[^A-z\s\-]*', '')) ))


                     .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
                     .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), dash_expr, ' '))

                        .withColumn('normalized_alt_names', func.transform(func.col('display_name_alternatives'),
                                    lambda x : func.regexp_replace(func.translate(func.lower(func.regexp_replace(x,  r'(?<=\s)[A-Z]\.*\s', '')), accented, plain), dash_expr, ' ')))

                        .withColumn('normalized_alt_names',  # remove author names that have nothing to do with actual profile to remove potential doubles
                                    func.array_compact(func.transform(func.col('normalized_alt_names'),
                                                                      lambda x: func.when(func.levenshtein(x, func.col('normalized_name')) < 
                                                                                          func.least(func.length(x), func.length(func.col('normalized_name'))),
                                                                                          x) )))
                       )

au_level_char_w_inst.cache()
au_level_char_w_inst.show()

+-----------+--------------------+--------------------+-------------------------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+--------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+
|  author_id|        display_name|               orcid|display_name_alternatives|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|         author_name|domain|field| subfield|    topic|author_first_name|gender|        affiliations|                city|associated_institutions|            acronyms|     normalized_name|normalized_alt_names|
+-----------+--------------------+--------------------+-------------------------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+--------------------+--------------------+-----------------------+--

In [8]:
au_level_char_w_inst.filter(func.col('author_id')=="A5078450184").show()

+-----------+--------------------+--------------------+-------------------------+----------+---------+---------------+------------+-------------+--------------------+------+-----+--------+-----+-----------------+------+--------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+
|  author_id|        display_name|               orcid|display_name_alternatives|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|         author_name|domain|field|subfield|topic|author_first_name|gender|        affiliations|                city|associated_institutions|            acronyms|     normalized_name|normalized_alt_names|
+-----------+--------------------+--------------------+-------------------------+----------+---------+---------------+------------+-------------+--------------------+------+-----+--------+-----+-----------------+------+--------------------+--------------------+-----------------------+-----------------

In [11]:
au_level_char_w_inst.filter(func.col('author_id')=="A5020571619").select(func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', ''))).show(truncate= False)

+----------------------------------------------------------+
|lower(regexp_replace(author_name, (?<=\s)[A-Z]\.*\s, , 1))|
+----------------------------------------------------------+
+----------------------------------------------------------+



In [12]:
anr_partenaires_2005 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-2005-2009-projets-finances-20251105-partenaires.csv', header = True, sep = ';')
                                           .withColumn('Projet.Partenaire.Code_RNSR', func.lit(''))

                       )

anr_partenaires_2010 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-depuis-2010-projets-finances-20260402-partenaires.csv', header = True, sep = ';')
                        )

anr_partenaires = (anr_partenaires_2005
                        .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2005.columns])
                   .select(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                   .unionAll(anr_partenaires_2010
                             .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                            )
                  )

In [13]:
anr_partenaires_2010

DataFrame[Projet.Code_Decision: string, Projet.Acronyme: string, Projet.Partenaire.Code_Decision: string, Projet.Partenaire.Est_coordinateur: string, Projet.Partenaire.Nom_organisme: string, Projet.Partenaire.Categorie_organisme: string, Projet.Partenaire.Code_RNSR: string, Projet.Partenaire.Responsable_scientifique.Nom: string, Projet.Partenaire.Responsable_scientifique.Prenom: string, Projet.Partenaire.Responsable_scientifique.ORCID: string, Projet.Partenaire.Adresse.Ville: string, Projet.Partenaire.Adresse.Region: string, Projet.Partenaire.Adresse.Pays: string, Projet.Partenaire.Aide_allouee.ANR: string, Projet.Partenaire.Financement.Source: string]

In [14]:

anr_partenaires = (anr_partenaires_2005
                        .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2005.columns])
                   .select(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                   .unionAll(anr_partenaires_2010
                             .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                            )
                  )

In [15]:
anr_partenaires.cache()
anr_partenaires.show()

+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+
|   Code_Decision|    Acronyme|    P_Code_Decision|P_Est_coordinateur|     P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsable_scientifique__Prenom|P_Responsable_scientifique__ORCID|P_Adresse__Ville|   P_Adresse__Region|P_Adresse__Pays|P_Aide_allouee__ANR|P_Financement__Source|
+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+
|ANR-09-VPTT-0015|ATAC-CONCEPT|

In [37]:
anr_partenaires.select(func.lower(func.concat_ws(' ', func.col('P_Responsable_scientifique__Prenom'),
                                                                             func.col('P_Responsable_scientifique__Nom'))).alias('normalized_name')).distinct().count()

58739

In [1621]:
anr_partenaires.select('P_Financement__Source').distinct().show(truncate= False)

+---------------------+
|P_Financement__Source|
+---------------------+
|ANR                  |
|Autre                |
|Fonds Propres        |
+---------------------+



In [1436]:
anr_partenaires.count()

102430

In [11]:
anr_partenaires.filter(func.col('P_Adresse__Pays')=="France").count()

83779

In [17]:
merged_orcid = (anr_partenaires
                .filter(func.col('P_Responsable_scientifique__ORCID').isNotNull())
                .join(au_level_char
                      .select('author_id', func.regexp_replace(func.col('orcid'), 'https://orcid.org/', '' ).alias('P_Responsable_scientifique__ORCID'))
                      , on = 'P_Responsable_scientifique__ORCID', how = 'inner')
               )
merged_orcid.count()

37768

In [1562]:
merged_orcid.select('P_Responsable_scientifique__ORCID').distinct().count()

19371

In [1644]:
merged_orcid.filter(func.col('author_id')=="A5078450184").show()

+---------------------------------+-------------+--------+---------------+------------------+---------------+---------------------+-----------+-------------------------------+----------------------------------+----------------+-----------------+---------------+-------------------+---------------------+---------+
|P_Responsable_scientifique__ORCID|Code_Decision|Acronyme|P_Code_Decision|P_Est_coordinateur|P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsable_scientifique__Prenom|P_Adresse__Ville|P_Adresse__Region|P_Adresse__Pays|P_Aide_allouee__ANR|P_Financement__Source|author_id|
+---------------------------------+-------------+--------+---------------+------------------+---------------+---------------------+-----------+-------------------------------+----------------------------------+----------------+-----------------+---------------+-------------------+---------------------+---------+
+---------------------------------+-------------+--------+

In [18]:

first_merge = (anr_partenaires
               .join(merged_orcid.select('P_Responsable_scientifique__ORCID'), on ='P_Responsable_scientifique__ORCID', how = 'left_anti')
               .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.col('P_Responsable_scientifique__Prenom'),
                                                                             func.col('P_Responsable_scientifique__Nom'))) )
               .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
               .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), dash_expr, ' '))

               .withColumn('affiliation_normalized',  func.lower(func.regexp_replace(func.col('P_Nom_organisme'),  r'[^A-z\s\-]*', '')) )
               .join(au_level_char_w_inst
                     .withColumn('normalized_name', func.explode(func.array_append(func.col('normalized_alt_names'), func.col('normalized_name'))))
                     , on = 'normalized_name', how = 'left')
               .distinct()
               .withColumn('n_match', func.size(func.collect_set('author_id').over(Window.partitionBy('normalized_name'))))
               .withColumn('common_af', func.array_max(func.transform(func.col('affiliations'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )
               .withColumn('common_city', func.array_max(func.transform(func.col('city'), 
                                                       lambda x : ( ( func.lower(x).contains(func.lower(func.col('P_Adresse__Ville')) ) 
                                                                     | func.lower(func.col('P_Adresse__Ville')).contains( func.lower(x) )) ).cast('int')
                                                      )) )
               .withColumn('associated_inst', func.array_max(func.transform(func.col('associated_institutions'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )
               .withColumn('acronym_match', func.array_max(func.transform(func.col('acronyms'), 
                                                       lambda x : ( x== func.col('affiliation_normalized') ).cast('int')
                                                      )) )

               .fillna(0,['common_af','common_city','associated_inst','acronym_match'])

               .withColumn('matched', ( (func.col('n_match')==1) | (func.col('common_af')==1) | (func.col('common_city')==1 ) | (func.col('associated_inst') == 1 ) ).cast('int'))
               .withColumn('max_matched', func.max('matched').over(Window.partitionBy('normalized_name')))
              )
first_merge.cache()
first_merge.groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1|51276|
|      0|42053|
+-------+-----+



In [19]:
first_merge = (first_merge
               .withColumn('city_recoded', func.regexp_extract(func.lower(func.col('P_Nom_organisme')),  r'(?<=université libre de )([A-z]+)', 1))
               .withColumn('city_recoded', func.regexp_extract(func.lower(func.col('P_Nom_organisme')),  r'(?<=université de )([A-z]+)', 1))
               .withColumn('city_recoded', func.when(func.col('city_recoded')== "", 
                                                     func.regexp_extract(func.lower(func.col('P_Nom_organisme')), r'(?<=université )([A-z]+)', 1))
                           .otherwise(func.col('city_recoded'))
                          )
               .withColumn('city_recoded', func.when(func.col('city_recoded')== "", 
                                                     func.regexp_extract(func.lower(func.col('P_Nom_organisme')), r'(?<=university of )([A-z]+)', 1))
                           .otherwise(func.col('city_recoded'))
                          )
               .withColumn('city_recoded', func.initcap(func.col('city_recoded')))
               .withColumn('common_city', func.array_max(func.transform(func.col('city'), 
                                                       lambda x : ( ( func.lower(x).contains(func.lower(func.col('P_Adresse__Ville')) ) 
                                                                     | func.lower(func.col('P_Adresse__Ville')).contains( func.lower(x) )) 
                                                                 | ( func.lower(x).contains(func.lower(func.col('city_recoded')) ) 
                                                                     | func.lower(func.col('city_recoded')).contains( func.lower(x) )) 
                                                                 | ( func.lower(x).contains(func.lower(func.col('P_Nom_organisme')) ) 
                                                                     | func.lower(func.col('P_Nom_organisme')).contains( func.lower(x) )) 

                                                                  ).cast('int')
                                                      )) )

               
               .fillna(0,['common_af','common_city','associated_inst','acronym_match'])
               .withColumn('matched', ( (func.col('n_match')==1) | (func.col('common_af')==1) | (func.col('common_city')==1 ) | (func.col('associated_inst') == 1 ) ).cast('int'))

              )

In [20]:
first_merge.groupBy('normalized_name').agg(func.max('matched').alias('matched')).groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1|29502|
|      0|12542|
+-------+-----+



In [21]:
first_merge.groupBy('normalized_name', 'P_Est_coordinateur').agg(func.max('matched').alias('matched')).groupBy('matched','P_Est_coordinateur').count().sort('P_Est_coordinateur','matched').show()

+-------+------------------+-----+
|matched|P_Est_coordinateur|count|
+-------+------------------+-----+
|      0|             False|11058|
|      1|             False|21822|
|      0|              True| 1724|
|      1|              True|11694|
+-------+------------------+-----+



In [27]:
first_merge.filter(func.col('P_Adresse__Pays')=="France").groupBy('normalized_name', 'P_Est_coordinateur').agg(func.max('matched').alias('matched')).groupBy('matched','P_Est_coordinateur').count().sort('P_Est_coordinateur','matched').show()

+-------+------------------+-----+
|matched|P_Est_coordinateur|count|
+-------+------------------+-----+
|      0|             False| 3338|
|      1|             False|16533|
|      0|              True| 1305|
|      1|              True|11227|
+-------+------------------+-----+



In [22]:
first_merge.filter(func.col('P_Responsable_scientifique__Nom')=='ALLONCLE').show()

+--------------------+---------------------------------+----------------+--------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+----------------+--------------------+---------------+-------------------+---------------------+----------------------+-----------+--------------------+--------------------+-------------------------+----------+---------+---------------+------------+-------------+--------------------+------+-----+--------+-----+-----------------+------+--------------------+--------------------+-----------------------+--------------------+--------------------+-------+---------+-----------+---------------+-------------+-------+-----------+------------+
|     normalized_name|P_Responsable_scientifique__ORCID|   Code_Decision|Acronyme|    P_Code_Decision|P_Est_coordinateur|     P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsa

In [23]:
unmatched = (anr_partenaires
               .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.col('P_Responsable_scientifique__Prenom'),
                                                                             func.col('P_Responsable_scientifique__Nom'))) )
               .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
             .join(first_merge
                   .filter('max_matched=0')
                   .select('normalized_name'), on = 'normalized_name', how = 'inner')
            )

In [24]:
unmatched.distinct().count()

23001

In [25]:
second_merge = (unmatched
                .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.substring(func.col('P_Responsable_scientifique__Prenom'), 1, 1),
                                                                         func.col('P_Responsable_scientifique__Nom'))))
                .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
               .join(au_level_char
                     .select('author_id', 'author_name',
                             func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', '')).alias('normalized_name') )
                     .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
                     .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), r'\.', ""))
                     , on = 'normalized_name', how = 'left')
                .withColumn('matched', (func.col('author_id').isNotNull()).cast('int'))
                .distinct()

               )
second_merge.groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1|17393|
|      0|18812|
+-------+-----+



In [26]:
second_merge.groupBy('normalized_name').agg(func.max('matched').alias('matched')).groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1| 3024|
|      0| 9860|
+-------+-----+



In [28]:
unmatched_r2 = (unmatched
                .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.substring(func.col('P_Responsable_scientifique__Prenom'), 1, 1),
                                                                         func.col('P_Responsable_scientifique__Nom'))))
                .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))

             .join(second_merge
                   .filter('matched=0')
                   .select('normalized_name').distinct(), on = 'normalized_name', how = 'inner')
            )
#unmatched_r2.count()

In [29]:
unmatched_r2.select('normalized_name').distinct().count()

9860

In [32]:
unmatched_r2.filter(func.col('P_Adresse__Pays')=="France").select('P_Responsable_scientifique__Prenom', 'P_Responsable_scientifique__Nom').distinct().show()

+----------------------------------+-------------------------------+
|P_Responsable_scientifique__Prenom|P_Responsable_scientifique__Nom|
+----------------------------------+-------------------------------+
|                            Daniel|                       BENOILID|
|                             Dormy|                       EMMANUEL|
|                               Dan|                        XU SONG|
|                        Abderrahim|               EL MOATAZ BILLAH|
|                              Anne|                 WEBER BENAROUS|
|                          Florence|                          PINET|
|                           Ibrahim|                        AL ASMI|
|                            Lionel|                         TAUNAY|
|                           Nicolas|                         GILLES|
|                             Serge|                       GRAVEROL|
|                              Yves|           ZEREGA (& CARETTE...|
|                        Aikaterin

In [35]:
au_level_char_w_inst.filter(func.col('author_id')=="A5066912127").show()

+---------+------------+-----+-------------------------+----------+---------+---------------+------------+-------------+-----------+------+-----+--------+-----+-----------------+------+------------+----+-----------------------+--------+---------------+--------------------+
|author_id|display_name|orcid|display_name_alternatives|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|author_name|domain|field|subfield|topic|author_first_name|gender|affiliations|city|associated_institutions|acronyms|normalized_name|normalized_alt_names|
+---------+------------+-----+-------------------------+----------+---------+---------------+------------+-------------+-----------+------+-----+--------+-----+-----------------+------+------------+----+-----------------------+--------+---------------+--------------------+
+---------+------------+-----+-------------------------+----------+---------+---------------+------------+-------------+-----------+------+-----+--------+-----+-----------------+

## final merge

In [36]:
all_matched_authors_dgds = (merged_orcid
                            .select('P_Responsable_scientifique__Prenom', 'P_Responsable_scientifique__Nom', 'author_id')
                            .unionAll(first_merge
                                      .filter('matched=1')
                                      .select('P_Responsable_scientifique__Prenom','P_Responsable_scientifique__Nom', 'author_id')
                                      .distinct()
                                     )
                            .unionAll(second_merge
                                      .filter('matched=1')
                                      .select('P_Responsable_scientifique__Prenom','P_Responsable_scientifique__Nom', 'author_id')
                                      .distinct()
                                     )
                           )
all_matched_authors_dgds.count()


99648

In [38]:
all_matched_authors_dgds .select('P_Responsable_scientifique__Prenom','P_Responsable_scientifique__Nom').distinct().count()

48279

In [39]:
anr_partenaires_matched = (anr_partenaires
                           .join(all_matched_authors_dgds, on = ['P_Responsable_scientifique__Prenom','P_Responsable_scientifique__Nom'], how = 'left')
                          )

In [43]:
anr_partenaires_matched

DataFrame[P_Responsable_scientifique__Prenom: string, P_Responsable_scientifique__Nom: string, Code_Decision: string, Acronyme: string, P_Code_Decision: string, P_Est_coordinateur: string, P_Nom_organisme: string, P_Categorie_organisme: string, P_Code_RNSR: string, P_Responsable_scientifique__ORCID: string, P_Adresse__Ville: string, P_Adresse__Region: string, P_Adresse__Pays: string, P_Aide_allouee__ANR: string, P_Financement__Source: string, author_id: string]

In [40]:
anr_projets_2005 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-2005-2009-projets-finances-20251105-projets.csv', header = True, sep = ';')
                                           .withColumn('Projet.Partenaire.Code_RNSR', func.lit(''))

                       )

anr_projets_2010 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-depuis-2010-projets-finances-20260402-projets.csv', header = True, sep = ';')
                        )

anr_projets = (anr_projets_2005
                        .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_projets_2005.columns])
                   .select(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_projets_2010.columns])
                   .unionAll(anr_projets_2010
                             .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_projets_2010.columns])
                            )
                  )

In [42]:
anr_projets.show()

+----------------+------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+------------------------------+---------------+
|   Code_Decision|AAP__Edition|    Acronyme|     Titre__Francais|Titre__Anglais|    Resume__Francais|     Resume__Anglais|Programme__Acronyme|Montant__AF__Aide_allouee__ANR|T0 scientifique|
+----------------+------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+------------------------------+---------------+
|ANR-09-VPTT-0015|        2009|ATAC-CONCEPT|Convertisseur aux...|          NULL|"L’objectif du Pr...|                NULL|                VTT|                      481372.6|     2009-12-01|
|ANR-09-VPTT-0014|        2009|     VOLHAND|VOLant pour perso...|          NULL|A ce jour, il n’e...|                NULL|                VTT|                     985433.14|     2009-10-01|
|ANR-09-VPTT-0013|        2009|     TAYLRUB|Gérer 

In [48]:
anr_dgds_all_info = (anr_partenaires_matched
                     .join(anr_projets.select('Code_Decision', *[col for col in anr_projets.columns if col not in anr_partenaires_matched.columns])
                           , on = 'Code_Decision', how = 'left')
                    )

In [49]:
anr_dgds_all_info.write.parquet(save_path + '/anr_dgds_all_info.parquet')

# ERC data

In [9]:
erc_grants = pd.read_excel("file:///" + save_path + "erc_export_grants.xlsx")

In [10]:
erc_grants = spark.createDataFrame(erc_grants)

In [11]:
erc_grants.show()

+--------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------+------------+--------------+-------------+-------------+---------+-------------------+-------------------+---------------+--------------------+
|     Programme|    Acronym|       Project Title|            Abstract|       Researcher(s)| Host Institution(s)|       Country|              Region|Project Number|        Call|    Grant Type|       Domain|        Panel|Call Year|         Start Date|           End Date|EU contribution|         CORDIS Link|
+--------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------+------------+--------------+-------------+-------------+---------+-------------------+-------------------+---------------+--------------------+
|Horizon Europe|  3Stops2Go|THREE STOP CODONS...|An accumulating b...|Julius Lu

In [55]:
erc_grants.count()

18521

In [57]:
erc_grants.filter(func.col('country')=='France').count()

2126

In [66]:
erc_grants.withColumn('Researcher', func.explode(func.split(func.col('Researcher(s)'),', '))).select("Researcher(s)", 'Researcher').show(truncate = False)

+-----------------------------------------------------------------------+---------------------+
|Researcher(s)                                                          |Researcher           |
+-----------------------------------------------------------------------+---------------------+
|Julius Lukes, Leos Valasek, Mark Osborn, Olivier Namy                  |Julius Lukes         |
|Julius Lukes, Leos Valasek, Mark Osborn, Olivier Namy                  |Leos Valasek         |
|Julius Lukes, Leos Valasek, Mark Osborn, Olivier Namy                  |Mark Osborn          |
|Julius Lukes, Leos Valasek, Mark Osborn, Olivier Namy                  |Olivier Namy         |
|Dale Abel, Guadalupe Sabio, Jesper Velgaard Olsen, Mauro Giacca        |Dale Abel            |
|Dale Abel, Guadalupe Sabio, Jesper Velgaard Olsen, Mauro Giacca        |Guadalupe Sabio      |
|Dale Abel, Guadalupe Sabio, Jesper Velgaard Olsen, Mauro Giacca        |Jesper Velgaard Olsen|
|Dale Abel, Guadalupe Sabio, Jesper Velg

In [75]:
erc_grants.withColumn('Researcher', func.explode(func.split(func.col('Researcher(s)'),', '))).select('Researcher').distinct().count()

15784

In [12]:
first_merge_erc = (erc_grants.withColumn('Researcher', func.explode(func.split(func.col('Researcher(s)'),', ')))
                   .withColumn('normalized_name', func.lower(func.col('Researcher')))
                   .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), r"\([a-z\s]*\)", ""))
                   .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
                   .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), dash_expr, ' '))
                   .withColumn('affiliation_normalized',  func.explode(func.split(func.lower(func.regexp_replace(func.col('Host Institution(s)'),  r'\[\d+\,[A-z]{2}\]', '')) , ', ')))
                   .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'university', 'universite'))
                   .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'institute', 'institut'))
                   .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'laboratory', 'laboratoire'))

                   .withColumn('affiliation_acronym',  func.regexp_extract(func.col('affiliation_normalized'), r'(?<=\()([a-z\-]*)(?=\))', 1))

                   
                   .join(au_level_char_w_inst
                     .withColumn('normalized_name', func.explode(func.array_append(func.col('normalized_alt_names'), func.col('normalized_name'))))
                     , on = 'normalized_name', how = 'left')
                   .withColumn('n_match', func.size(func.collect_set('author_id').over(Window.partitionBy('normalized_name'))))
                   .withColumn('common_af', func.array_max(func.transform(func.col('affiliations'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )
                   .withColumn('common_acronym', func.array_max(func.transform(func.col('acronyms'), 
                                                       lambda x : ( x == func.col('affiliation_acronym') ).cast('int')
                                                      )) )

                   .fillna(0, subset = ['common_af','common_acronym'])
                   .withColumn('matched', ( (func.col('n_match')==1) | (func.col('common_af')==1) | (func.col('common_acronym')==1) ).cast('int'))

                  )

               
               

first_merge_erc.cache()


DataFrame[normalized_name: string, Programme: string, Acronym: string, Project Title: string, Abstract: string, Researcher(s): string, Host Institution(s): string, Country: string, Region: string, Project Number: bigint, Call: string, Grant Type: string, Domain: string, Panel: string, Call Year: bigint, Start Date: timestamp, End Date: timestamp, EU contribution: double, CORDIS Link: string, Researcher: string, affiliation_normalized: string, affiliation_acronym: string, author_id: string, display_name: string, orcid: string, display_name_alternatives: array<string>, entry_year: int, last_year: int, all_y_in_sample: bigint, years_abroad: bigint, entry_country: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string, affiliations: array<string>, city: array<string>, associated_institutions: array<string>, acronyms: array<string>, normalized_alt_names: array<string>, n_match: int, common_af: int, common_acrony

In [13]:
first_merge_erc = (first_merge_erc                   .withColumn('affiliation_normalized',  func.explode(func.split(func.lower(func.regexp_replace(func.col('Host Institution(s)'),  r'\[\d+\,[A-z]{2}\]', '')) , ', ')))
                  .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'university', 'universite'))
                   .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'institute', 'institut'))
                   .withColumn('affiliation_normalized', func.regexp_replace(func.col('affiliation_normalized'), 'laboratory', 'laboratoire'))
                   .withColumn('common_af', func.array_max(func.transform(func.col('affiliations'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )
                   .withColumn('common_acronym', func.array_max(func.transform(func.col('acronyms'), 
                                                       lambda x : ( x == func.col('affiliation_acronym') ).cast('int')
                                                      )) )

                   .fillna(0, subset = ['common_af','common_acronym'])
                   .withColumn('matched', ( (func.col('n_match')==1) | (func.col('common_af')==1) | (func.col('common_acronym')==1) ).cast('int'))
                  )


first_merge_erc.cache()


DataFrame[normalized_name: string, Programme: string, Acronym: string, Project Title: string, Abstract: string, Researcher(s): string, Host Institution(s): string, Country: string, Region: string, Project Number: bigint, Call: string, Grant Type: string, Domain: string, Panel: string, Call Year: bigint, Start Date: timestamp, End Date: timestamp, EU contribution: double, CORDIS Link: string, Researcher: string, affiliation_normalized: string, affiliation_acronym: string, author_id: string, display_name: string, orcid: string, display_name_alternatives: array<string>, entry_year: int, last_year: int, all_y_in_sample: bigint, years_abroad: bigint, entry_country: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string, affiliations: array<string>, city: array<string>, associated_institutions: array<string>, acronyms: array<string>, normalized_alt_names: array<string>, n_match: int, common_af: int, common_acrony

In [14]:
first_merge_erc.groupBy( (func.col('country')=='France').cast('int').alias('Fr'), 'normalized_name').agg(func.max('matched').alias('matched')).groupBy('Fr', 'matched').count().show()


+---+-------+-----+
| Fr|matched|count|
+---+-------+-----+
|  1|      0|  616|
|  1|      1| 1370|
|  0|      0| 8245|
|  0|      1| 5481|
+---+-------+-----+



In [112]:
first_merge_erc.groupBy('Researcher', (func.col('country')=='France').cast('int').alias('Fr'), 'normalized_name').agg(func.max('matched').alias('matched')).filter('matched=0').filter('Fr=1').select('Researcher').toPandas().to_csv(save_path + "erc_au_a_matcher.csv")

In [19]:
erc_matched_hand = spark.createDataFrame(pd.read_excel(save_path + "erc_matched_hand.xlsx")[['Researcher', 'matched', 'author_id']])

In [20]:
erc_matched_hand.show()

+--------------------+-------+-----------+
|          Researcher|matched|  author_id|
+--------------------+-------+-----------+
| Anaëlle Julie Maury|    1.0|A5018695775|
|    Boris Elie Burle|    1.0|A5083214721|
|Bruno Eric Emmanu...|    1.0|A5038644278|
|Jean-Bernard Lass...|    1.0|A5006670993|
|Pierre Marie Vict...|    1.0|A5014081829|
|Antonio Angelo Ba...|    1.0|A5078691950|
|Dierk Sebastian S...|    1.0|A5059255533|
|          Elise Pape|    1.0|A5034621661|
|Hugo Denis Misaël...|    1.0|A5065401344|
|Jean-Baptiste Bru...|    1.0|A5073537354|
|Anabela Ben-Simon...|    1.0|A5046019304|
|  John Kevin O'Regan|    1.0|A5006244844|
|Michaël Philippe ...|    1.0|A5010500477|
|Nicolas Reyes Garcia|    1.0|A5065205905|
|     Nikolai Chapiro|    1.0|A5031082257|
|Pascale Françoise...|    1.0|A5065282571|
|Pascale Francoise...|    1.0|A5065282571|
|  Raymond Demadrille|    1.0|A5039936260|
|              Renaud|    NaN|        NaN|
|      Anne-Madeleine|    NaN|        NaN|
+----------

In [22]:
erc_matched = (first_merge_erc
               .filter('matched=1')
               .select('Researcher', 'CORDIS Link', 'author_id')
               .distinct()
               .unionAll(erc_matched_hand
                         .join(first_merge_erc.select('Researcher','CORDIS Link').distinct(),
                               on = 'Researcher', how = 'left')
                         .select('Researcher', 'CORDIS Link', 'author_id')
                        )
              )
erc_matched.count()

9860

In [23]:
erc_matched_all_info = (erc_grants
                        .join(erc_matched
                              .groupBy('CORDIS Link')
                              .agg(func.collect_set('author_id').alias('author_id')),
                              on = 'CORDIS Link', how = 'inner')
                       )
erc_matched_all_info.count()

8902

In [24]:
erc_matched_all_info.write.parquet(save_path + '/erc_matched_all_info.parquet')

In [25]:
spark.catalog.clearCache()